**This notebook is an exercise in the [Advanced SQL](https://www.kaggle.com/learn/advanced-sql) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/analytic-functions).**

---


# Introduction

Here, you'll use window functions to answer questions about the [Chicago Taxi Trips](https://www.kaggle.com/chicago/chicago-taxi-trips-bq) dataset.

Before you get started, run the code cell below to set everything up.

In [1]:
# Get most recent checking code
!pip install -U -t /kaggle/working/ git+https://github.com/Kaggle/learntools.git
# Set up feedback system
from learntools.core import binder
binder.bind(globals())
from learntools.sql_advanced.ex2 import *
print("Setup Complete")

  Cloning https://github.com/Kaggle/learntools.git to /tmp/pip-req-build-k8cfafiz
  Running command git clone --filter=blob:none --quiet https://github.com/Kaggle/learntools.git /tmp/pip-req-build-k8cfafiz
  Resolved https://github.com/Kaggle/learntools.git to commit 8a8d4450a699c864febc419fb27ec41443a5942b
  Preparing metadata (setup.py) ... done
  Created wheel for learntools: filename=learntools-0.3.5-py3-none-any.whl size=269683 sha256=dcf3c8b8cbd2e590e73e69b59914127b0c87324ef67479e7b208c88e268c0d38
  Stored in directory: /tmp/pip-ephem-wheel-cache-9ps8nb3r/wheels/50/16/ba/3e0ec276f3238de9e1c59751f34c7c3ea4a3f561af10c4fd0c
Successfully built learntools
Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Setup Complete


The following code cell fetches the `taxi_trips` table from the `chicago_taxi_trips` dataset. We also preview the first five rows of the table.  You'll use the table to answer the questions below.

In [2]:
from google.cloud import bigquery

# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "chicago_taxi_trips" dataset
dataset_ref = client.dataset("chicago_taxi_trips", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "taxi_trips" table
table_ref = dataset_ref.table("taxi_trips")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=5).to_dataframe()

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,unique_key,taxi_id,trip_start_timestamp,trip_end_timestamp,trip_seconds,trip_miles,pickup_census_tract,dropoff_census_tract,pickup_community_area,dropoff_community_area,...,extras,trip_total,payment_type,company,pickup_latitude,pickup_longitude,pickup_location,dropoff_latitude,dropoff_longitude,dropoff_location
0,b99416692f5bc01cdb444cbcec7e1105d30d455d,8dd990653cc2793d96b80ae45e928afc9a590b21b491ee...,2019-05-29 18:00:00+00:00,2019-05-29 18:30:00+00:00,1500,1.50,<NA>,<NA>,<NA>,<NA>,...,0.0,12.36,Credit Card,KOAM Taxi Association,NaN,NaN,None,NaN,NaN,None
1,b99887012c8beccc1dae3f6fcde2fc94ff0ef7b1,d156d4adff01addd62b65753ce67dc8e04bdaf60f7eb14...,2019-05-20 19:15:00+00:00,2019-05-20 19:30:00+00:00,506,1.67,<NA>,<NA>,<NA>,<NA>,...,0.0,10.50,Credit Card,Chicago Carriage Cab Corp,NaN,NaN,None,NaN,NaN,None
2,b98b6b7afe1f8065d8ba71cd023d9a13f2460f84,beefd3462e3f5a8e854942a2796876f6db73ebbd25b435...,2019-05-08 00:00:00+00:00,2019-05-08 00:00:00+00:00,60,0.00,<NA>,<NA>,<NA>,<NA>,...,0.0,3.25,Cash,Taxi Affiliation Services,NaN,NaN,None,NaN,NaN,None
3,b9865af3673441b1a79973ad08bd9909bd422af4,ef249ea2c31ccb20ce9eb258d3116254a6f9fe05c7cc84...,2019-05-07 16:30:00+00:00,2019-05-07 17:00:00+00:00,1122,2.86,<NA>,<NA>,<NA>,<NA>,...,1.0,13.25,Cash,Taxi Affiliation Service Yellow,NaN,NaN,None,NaN,NaN,None
4,e04dbd9a8ba723cc785c4b0398106b72e16d7870,45002471b85d61e62cf602a15145d2bff912e080f50470...,2013-05-16 08:15:00+00:00,2013-05-16 08:15:00+00:00,<NA>,0.00,<NA>,<NA>,<NA>,<NA>,...,0.0,30.06,Credit Card,Chicago Elite Cab Corp.,NaN,NaN,None,NaN,NaN,None


# Exercises

### 1) How can you predict the demand for taxis?

Say you work for a taxi company, and you're interested in predicting the demand for taxis.  Towards this goal, you'd like to create a plot that shows a rolling average of the daily number of taxi trips.  Amend the (partial) query below to return a DataFrame with two columns:
- `trip_date` - contains one entry for each date from January 1, 2016, to March 31, 2016.
- `avg_num_trips` - shows the average number of daily trips, calculated over a window including the value for the current date, along with the values for the preceding 3 days and the following 3 days, as long as the days fit within the three-month time frame.  For instance, when calculating the value in this column for January 3, 2016, the window will include the number of trips for the preceding 2 days, the current date, and the following 3 days.

This query is partially completed for you, and you need only write the part that calculates the `avg_num_trips` column.  Note that this query uses a common table expression (CTE); if you need to review how to use CTEs, you're encouraged to check out [this tutorial](https://www.kaggle.com/dansbecker/as-with) in the [Intro to SQL](https://www.kaggle.com/learn/intro-to-sql) course.

In [3]:
# Fill in the blank below
avg_num_trips_query = """
WITH trips_by_day AS
(
    SELECT DATE(trip_start_timestamp) AS trip_date,
           COUNT(*) AS num_trips
    FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
    WHERE trip_start_timestamp > '2016-01-01'
      AND trip_start_timestamp < '2016-04-01'
    GROUP BY trip_date
    ORDER BY trip_date
)
SELECT trip_date,
       AVG(num_trips) OVER() AS avg_num_trips
FROM trips_by_day
"""

# Check your answer
q_1.check()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,trip_date,avg_num_trips
0,2016-02-29,87131.230769
1,2016-02-21,87131.230769
2,2016-03-21,87131.230769
3,2016-03-06,87131.230769
4,2016-03-09,87131.230769


<IPython.core.display.Javascript object>

<span style="color:#cc3333">Incorrect:</span> Your window should include the current date, along with the preceding 3 days and the following 3 days.

In [4]:
# Lines below will give you a hint or solution code
#q_1.hint()
#q_1.solution()

### 2) Can you separate and order trips by community area?

The query below returns a DataFrame with three columns from the table: `pickup_community_area`, `trip_start_timestamp`, and `trip_end_timestamp`.  

Amend the query to return an additional column called `trip_number` which shows the order in which the trips were taken from their respective community areas.  So, the first trip of the day originating from community area 1 should receive a value of 1; the second trip of the day from the same area should receive a value of 2.  Likewise, the first trip of the day from community area 2 should receive a value of 1, and so on.

Note that there are many numbering functions that can be used to solve this problem (depending on how you want to deal with trips that started at the same time from the same community area); to answer this question, please use the **RANK()** function.

In [5]:
# Amend the query below
trip_number_query = """
                    SELECT pickup_community_area,
                        trip_start_timestamp,
                        trip_end_timestamp,
                        RANK() OVER (
                            PARTITION BY pickup_community_area
                            ORDER BY trip_start_timestamp
                        ) AS trip_number
                    FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                    WHERE DATE(trip_start_timestamp) = '2013-10-03'
                    """

# Check your answer
q_2.check()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,pickup_community_area,trip_start_timestamp,trip_end_timestamp,trip_number
0,63,2013-10-03 06:45:00+00:00,2013-10-03 07:00:00+00:00,1
1,63,2013-10-03 08:45:00+00:00,2013-10-03 08:45:00+00:00,2
2,19,2013-10-03 00:00:00+00:00,2013-10-03 00:00:00+00:00,1
3,19,2013-10-03 00:00:00+00:00,2013-10-03 00:00:00+00:00,1
4,19,2013-10-03 03:30:00+00:00,2013-10-03 04:15:00+00:00,3


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [6]:
# Lines below will give you a hint or solution code
#q_2.hint()
#q_2.solution()

### 3) How much time elapses between trips?

The (partial) query in the code cell below shows, for each trip in the selected time frame, the corresponding `taxi_id`, `trip_start_timestamp`, and `trip_end_timestamp`. 

Your task in this exercise is to edit the query to include an additional `prev_break` column that shows the length of the break (in minutes) that the driver had before each trip started (this corresponds to the time between `trip_start_timestamp` of the current trip and `trip_end_timestamp` of the previous trip).  Partition the calculation by `taxi_id`, and order the results within each partition by `trip_start_timestamp`.

Some sample results are shown below, where all rows correspond to the same driver (or `taxi_id`).  Take the time now to make sure that the values in the `prev_break` column make sense to you!

![first_commands](https://storage.googleapis.com/kaggle-media/learn/images/qjvQzg8.png)

Note that the first trip of the day for each driver should have a value of **NaN** (not a number) in the `prev_break` column.

In [7]:
# Fill in the blanks below
break_time_query = """
                   SELECT taxi_id,
                       trip_start_timestamp,
                       trip_end_timestamp,
                       TIMESTAMP_DIFF(
                           trip_start_timestamp, 
                           LAG(trip_end_timestamp)
                               OVER (
                                    PARTITION BY taxi_id
                                    ORDER BY trip_start_timestamp), 
                           MINUTE) as prev_break
                   FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                   WHERE DATE(trip_start_timestamp) = '2013-10-03' 
                   """

# Check your answer
q_3.check()

/usr/local/lib/python3.11/dist-packages/google/cloud/bigquery/table.py:1727: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,taxi_id,trip_start_timestamp,trip_end_timestamp,prev_break
0,01fea6dd04ff9ae6c26502b2742bb6d4e06be62d2c0095...,2013-10-03 03:30:00+00:00,2013-10-03 03:30:00+00:00,-30
1,01fea6dd04ff9ae6c26502b2742bb6d4e06be62d2c0095...,2013-10-03 09:00:00+00:00,2013-10-03 09:30:00+00:00,330
2,01fea6dd04ff9ae6c26502b2742bb6d4e06be62d2c0095...,2013-10-03 19:45:00+00:00,2013-10-03 19:45:00+00:00,255
3,01fea6dd04ff9ae6c26502b2742bb6d4e06be62d2c0095...,2013-10-03 21:30:00+00:00,2013-10-03 21:30:00+00:00,-30
4,171f9117ef362705e619fa2dbf6100330ac45052849edf...,2013-10-03 15:45:00+00:00,2013-10-03 16:15:00+00:00,840


<IPython.core.display.Javascript object>

<span style="color:#33cc33">Correct</span>

In [8]:
# Lines below will give you a hint or solution code
#q_3.hint()
#q_3.solution()

# Keep going

Move on to learn how to query **[nested and repeated data](https://www.kaggle.com/alexisbcook/nested-and-repeated-data)**.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/advanced-sql/discussion) to chat with other learners.*